# 05x feature contract rebuild patch 260515

Patch-only step. No modeling, EDA, SHAP, Optuna, or segmentation is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import zipfile

import pandas as pd

ROOT = Path(r'C:/Code/ott-churn-prediction').resolve()
PARK = ROOT / 'park.ingyeom'
PREV_DIR = PARK / 'reports' / 'audits' / '05x_feature_contract_rebuild_260515'
PREV_NB = PARK / 'notebook' / '05x_feature_contract_rebuild_260515' / '05x_feature_contract_rebuild_260515.ipynb'
SOURCE_MASTER = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
NOTE = PARK / 'note.md'
OUT_DIR = PARK / 'reports' / 'audits' / '05x_feature_contract_rebuild_patch_260515'
NB_OUT = PARK / 'notebook' / '05x_feature_contract_rebuild_patch_260515' / '05x_feature_contract_rebuild_patch_260515.ipynb'
ZIP_PATH = PARK / 'zip' / '05x_feature_contract_rebuild_patch_260515_review_package.zip'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

prev_decision_path = PREV_DIR / '05x_feature_resolution_decision_table.csv'
prev_preflight_path = PREV_DIR / '05x_preflight_input_validation.csv'
decision = pd.read_csv(prev_decision_path)
source_cols = list(pd.read_csv(SOURCE_MASTER, nrows=0).columns)

valid_decisions = {
    'keep_in_conservative_safe_22',
    'candidate_for_expanded_feature_set',
    'candidate_for_expanded_with_caveat',
    'forbidden_or_audit_only_candidate',
    'unresolved_user_review_required',
}
mandatory = ['column_name', 'llm_proposed_decision', 'reason', 'user_approval_required', 'final_decision_status']
decision_before = decision.copy()

def set_row(col, **values):
    mask = decision['column_name'].eq(col)
    if mask.any():
        for key, value in values.items():
            if key not in decision.columns:
                decision[key] = ''
            decision.loc[mask, key] = value
    return bool(mask.any())

for col in mandatory:
    if col not in decision.columns:
        decision[col] = ''

set_row(
    'USER_KEY',
    llm_proposed_decision='forbidden_or_audit_only_candidate',
    reason='row identifier / group key only, not a model feature',
    user_approval_required='no',
    final_decision_status='fixed_by_policy_not_model_feature',
    caution='policy-fixed audit/group key only',
)
set_row(
    'is_repurchase',
    llm_proposed_decision='forbidden_or_audit_only_candidate',
    reason='target variable, never a model feature',
    user_approval_required='no',
    final_decision_status='fixed_by_policy_not_model_feature',
    caution='policy-fixed target leakage prohibition',
)

price_screen_caution = '과거 사용자 결정과 충돌 가능성 있음. price/max_screen은 사용자가 제거 또는 대체를 언급한 이력이 있으므로 자동 expanded 승인 금지'
for col in ['price', 'max_screen']:
    set_row(
        col,
        llm_proposed_decision='unresolved_user_review_required',
        user_approval_required='yes',
        caution=price_screen_caution,
        final_decision_status='pending_user_approval',
        reason='requires explicit user review before any expanded feature approval',
    )

invalid_mask = ~decision['llm_proposed_decision'].isin(valid_decisions)
decision.loc[invalid_mask, 'reason'] = decision.loc[invalid_mask, 'reason'].fillna('')
decision.loc[invalid_mask, 'llm_proposed_decision'] = 'unresolved_user_review_required'
decision.loc[invalid_mask, 'user_approval_required'] = 'yes'
decision.loc[invalid_mask, 'final_decision_status'] = 'pending_user_approval'

for col in mandatory:
    blank = decision[col].isna() | decision[col].astype(str).str.strip().eq('')
    if col == 'user_approval_required':
        decision.loc[blank, col] = 'yes'
    elif col == 'final_decision_status':
        decision.loc[blank, col] = 'pending_user_approval'
    elif col == 'reason':
        decision.loc[blank, col] = 'carried forward from previous 05x contract; requires user review if not policy-fixed'
    else:
        decision.loc[blank, col] = 'unknown'

decision.to_csv(OUT_DIR / '05x_feature_resolution_decision_table.csv', index=False, encoding='utf-8-sig')

preflight_prev = pd.read_csv(prev_preflight_path)
archive_row = preflight_prev.loc[preflight_prev['check'].eq('archive_reference_exists')]
archive_prev_detail = archive_row['detail'].iloc[0] if len(archive_row) else 'not recorded'
archive_prev_status = archive_row['status'].iloc[0] if len(archive_row) else 'WARN'
archive_exists = str(archive_prev_detail).strip().lower() == 'true'
preflight = pd.DataFrame([
    {'check': 'source_master_exists', 'status': 'PASS' if SOURCE_MASTER.exists() else 'FAIL', 'detail': str(SOURCE_MASTER)},
    {'check': 'previous_05x_folder_exists', 'status': 'PASS' if PREV_DIR.exists() else 'FAIL', 'detail': str(PREV_DIR)},
    {'check': 'previous_05x_notebook_exists', 'status': 'PASS' if PREV_NB.exists() else 'FAIL', 'detail': str(PREV_NB)},
    {'check': 'previous_decision_table_exists', 'status': 'PASS' if prev_decision_path.exists() else 'FAIL', 'detail': str(prev_decision_path)},
    {'check': 'archive_reference_exists', 'status': 'PASS' if archive_exists else 'WARN', 'detail': f'previous 05x preflight status={archive_prev_status}, detail={archive_prev_detail}; optional archive reference not used for this patch'},
    {'check': 'patch_output_folder', 'status': 'PASS' if OUT_DIR.exists() else 'FAIL', 'detail': str(OUT_DIR)},
    {'check': 'stop_reason', 'status': 'PASS', 'detail': 'none'},
])
preflight.to_csv(OUT_DIR / '05x_patch_preflight_validation.csv', index=False, encoding='utf-8-sig')

special_rows = []
for col in ['USER_KEY', 'is_repurchase', 'price', 'max_screen']:
    exists = col in set(decision['column_name'])
    if col in ['USER_KEY', 'is_repurchase'] and exists:
        action = 'policy fixed as forbidden_or_audit_only_candidate'
        status = 'patched'
    elif col in ['price', 'max_screen'] and exists:
        action = 'flagged as unresolved_user_review_required'
        status = 'patched'
    else:
        action = 'recorded as missing; no inferred substitute used'
        status = 'missing'
    special_rows.append({'special_case': col, 'column_exists': 'yes' if exists else 'no', 'action': action, 'status': status, 'note': 'actual column name checked only'})
special_rows.append({'special_case': 'archive_reference', 'column_exists': 'n/a', 'action': 'document previous WARN False and proceed from previous 05x outputs plus source master', 'status': 'warn_documented', 'note': 'archive reference was optional for patch; warning not hidden'})
pd.DataFrame(special_rows).to_csv(OUT_DIR / '05x_patch_special_case_audit.csv', index=False, encoding='utf-8-sig')

approval = decision[decision['user_approval_required'].astype(str).str.lower().eq('yes')].copy()
approval['user_decision'] = 'pending'
approval['review_note'] = 'user approval required before 06x'
approval.to_csv(OUT_DIR / '05x_user_approval_checklist.csv', index=False, encoding='utf-8-sig')

cons = decision[decision['llm_proposed_decision'].eq('keep_in_conservative_safe_22')].copy()
cons['contract_status'] = 'contract_candidate_pending_user_approval'
cons['plan'] = 'conservative_safe_22_contract_candidate'
cons.to_csv(OUT_DIR / '05x_conservative_safe_22_contract.csv', index=False, encoding='utf-8-sig')

expanded_codes = ['keep_in_conservative_safe_22', 'candidate_for_expanded_feature_set', 'candidate_for_expanded_with_caveat']
expanded = decision[decision['llm_proposed_decision'].isin(expanded_codes)].copy()
expanded['candidate_status'] = 'candidate_pending_user_approval'
expanded['context_content_caveat'] = expanded.apply(lambda r: str(r.get('feature_family', '')).lower().find('content') >= 0 or str(r.get('feature_family', '')).lower().find('context') >= 0 or str(r.get('llm_proposed_decision', '')).endswith('caveat'), axis=1)
expanded['plan'] = 'expanded_feature_set_candidate_contract'
expanded.to_csv(OUT_DIR / '05x_expanded_feature_set_candidate_contract.csv', index=False, encoding='utf-8-sig')

forbidden_mask = decision['llm_proposed_decision'].eq('forbidden_or_audit_only_candidate')
forbidden_mask |= decision['feature_family'].astype(str).str.contains('target|identifier|score', case=False, na=False)
forbidden_mask |= decision['leakage_risk'].astype(str).str.contains('high|leakage', case=False, na=False)
forbidden = decision[forbidden_mask].copy()
forbidden['status_note'] = 'policy model-feature prohibition or audit-only candidate; not an LLM final deletion'
forbidden.to_csv(OUT_DIR / '05x_forbidden_or_audit_only_candidates.csv', index=False, encoding='utf-8-sig')

unresolved = decision[decision['llm_proposed_decision'].eq('unresolved_user_review_required')].copy()
unresolved['user_decision'] = 'pending'
unresolved.to_csv(OUT_DIR / '05x_unresolved_user_review_required.csv', index=False, encoding='utf-8-sig')

diff_rows = []
track_cols = ['llm_proposed_decision', 'reason', 'user_approval_required', 'final_decision_status', 'caution']
for _, before in decision_before.iterrows():
    colname = before['column_name']
    after = decision.loc[decision['column_name'].eq(colname)].iloc[0]
    for c in track_cols:
        prev = '' if pd.isna(before.get(c, '')) else str(before.get(c, ''))
        new = '' if pd.isna(after.get(c, '')) else str(after.get(c, ''))
        if prev != new:
            diff_rows.append({'column_name': colname, 'changed_column': c, 'previous_value': prev, 'patched_value': new, 'reason': '05x patch requirement or mandatory blank normalization'})
pd.DataFrame(diff_rows).to_csv(OUT_DIR / '05x_patch_diff_summary.csv', index=False, encoding='utf-8-sig')

readme_text = f"""# 05x feature contract rebuild patch 260515

## Patch purpose
This patch corrects specific review findings in the previous 05x outputs without rebuilding the full 05x analysis.
No modeling, EDA, SHAP, Optuna, or segmentation was performed.

## Previous 05x issues found
- The review zip contained a notebook that was not an executed saved copy.
- `USER_KEY` and `is_repurchase` had reason sentences in `llm_proposed_decision` instead of approved decision codes.
- `USER_KEY` and `is_repurchase` had blank `user_approval_required` values.
- `price` and `max_screen` were expanded candidates, but may conflict with prior user decisions.
- Previous preflight recorded `archive_reference_exists = WARN False`.

## Patch changes
- Rebuilt the decision table as a patched 91-column contract copy.
- Restricted `llm_proposed_decision` values to the approved decision-code set.
- Removed blanks from key columns: `column_name`, `llm_proposed_decision`, `reason`, `user_approval_required`, `final_decision_status`.
- Wrote patch audit, approval checklist, candidate contracts, forbidden/audit-only candidates, unresolved review table, diff summary, final checks, and review zip.

## USER_KEY / is_repurchase result
- `USER_KEY`: `forbidden_or_audit_only_candidate`, policy fixed as row identifier / group key only, not a model feature.
- `is_repurchase`: `forbidden_or_audit_only_candidate`, policy fixed as target variable, never a model feature.

## price / max_screen user review
- `price` exists and is marked `unresolved_user_review_required` with `user_approval_required=yes`.
- `max_screen` exists and is marked `unresolved_user_review_required` with `user_approval_required=yes`.
- Caution: 과거 사용자 결정과 충돌 가능성 있음. price/max_screen은 사용자가 제거 또는 대체를 언급한 이력이 있으므로 자동 expanded 승인 금지

## archive_reference WARN
Previous 05x preflight recorded `archive_reference_exists` as `{archive_prev_status} {archive_prev_detail}`.
The patch does not hide this warning. The patch was performed from the existing 05x outputs and source master; the archive reference was an optional reference.

## Before 06x
The final feature-use decision is not complete until the user reviews the approval checklist.
Next step: review `05x_user_approval_checklist.csv`, then decide whether 06x may proceed.
"""
(OUT_DIR / 'README.md').write_text(readme_text, encoding='utf-8')

note_add = f"""

## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | 05x_feature_contract_rebuild_patch_260515

- 05x patch 수행.
- 기존 05x의 decision table 오류 수정.
- USER_KEY와 is_repurchase는 모델 feature 금지로 정책상 고정.
- price/max_screen은 사용자 확인 필요 항목으로 표시.
- 05x patch 이후에도 최종 feature 사용 여부는 사용자 승인 전까지 확정 아님.
- 06x는 사용자 승인 후 진행.
- output_dir: {OUT_DIR}
- review_zip: {ZIP_PATH}
"""
note_text_before = NOTE.read_text(encoding='utf-8') if NOTE.exists() else ''
if '05x_feature_contract_rebuild_patch_260515' not in note_text_before:
    NOTE.write_text(note_text_before.rstrip() + note_add, encoding='utf-8')
(OUT_DIR / 'note_tail_copy.md').write_text('\n'.join(NOTE.read_text(encoding='utf-8').splitlines()[-80:]), encoding='utf-8')

def has_executed_outputs(nb_path):
    if not nb_path.exists():
        return False
    data = json.loads(nb_path.read_text(encoding='utf-8'))
    code_cells = [c for c in data.get('cells', []) if c.get('cell_type') == 'code']
    return any(c.get('execution_count') is not None for c in code_cells) and any(c.get('outputs') for c in code_cells)

checks = []
def add_check(name, ok, detail):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': str(detail)})

inside = all(str(p.resolve()).startswith(str(PARK.resolve())) for p in [OUT_DIR, NB_OUT, ZIP_PATH])
add_check('all_outputs_inside_park_ingyeom', inside, PARK)
add_check('raw_source_csv_not_modified', True, 'read-only nrows=0 schema check only')
add_check('notebook_exists', NB_OUT.exists(), NB_OUT)
add_check('notebook_executed_with_outputs', has_executed_outputs(NB_OUT), 'checked execution_count and outputs in saved notebook')
add_check('previous_05x_loaded', len(decision_before) == 91, len(decision_before))
add_check('all_91_columns_present_in_decision_table', len(decision) == 91 and set(decision['column_name']) == set(source_cols), len(decision))
add_check('llm_proposed_decision_values_valid', set(decision['llm_proposed_decision']).issubset(valid_decisions), sorted(set(decision['llm_proposed_decision'])))
add_check('USER_KEY_decision_code_fixed', decision.loc[decision['column_name'].eq('USER_KEY'), 'llm_proposed_decision'].iloc[0] == 'forbidden_or_audit_only_candidate', 'USER_KEY')
add_check('is_repurchase_decision_code_fixed', decision.loc[decision['column_name'].eq('is_repurchase'), 'llm_proposed_decision'].iloc[0] == 'forbidden_or_audit_only_candidate', 'is_repurchase')
key_blank_count = int(decision[mandatory].isna().sum().sum() + decision[mandatory].astype(str).apply(lambda s: s.str.strip().eq('')).sum().sum())
add_check('no_blank_user_approval_required_for_key_columns', key_blank_count == 0, key_blank_count)
pm_ok = True
for col in ['price', 'max_screen']:
    if col in set(decision['column_name']):
        row = decision.loc[decision['column_name'].eq(col)].iloc[0]
        pm_ok = pm_ok and row['llm_proposed_decision'] == 'unresolved_user_review_required' and row['user_approval_required'] == 'yes'
add_check('price_max_screen_flagged_if_present', pm_ok, 'actual columns only')
add_check('archive_reference_warn_documented', True, f'previous 05x: {archive_prev_status} {archive_prev_detail}')
add_check('no_modeling_performed', True, 'patch-only notebook')
add_check('no_eda_performed', True, 'patch-only notebook')
add_check('no_shap_performed', True, 'patch-only notebook')
add_check('no_optuna_performed', True, 'patch-only notebook')
add_check('no_segmentation_performed', True, 'patch-only notebook')
add_check('README_created', (OUT_DIR / 'README.md').exists(), OUT_DIR / 'README.md')
add_check('note_md_updated', '05x_feature_contract_rebuild_patch_260515' in NOTE.read_text(encoding='utf-8'), NOTE)
add_check('review_zip_created', False, ZIP_PATH)
final_checks = pd.DataFrame(checks)
crit_fail_count = int((final_checks['status'] == 'FAIL').sum())
final_checks = pd.concat([final_checks, pd.DataFrame([{'check': 'critical_fail_count_zero', 'status': 'PASS' if crit_fail_count == 0 else 'FAIL', 'detail': str(crit_fail_count)}])], ignore_index=True)
final_checks.to_csv(OUT_DIR / '05x_final_checks.csv', index=False, encoding='utf-8-sig')

zip_members = [NB_OUT]
zip_members += sorted(OUT_DIR.glob('*.csv'))
zip_members += [OUT_DIR / 'README.md', OUT_DIR / 'note_tail_copy.md']
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in zip_members:
        if path.exists():
            zf.write(path, path.relative_to(PARK).as_posix())

final_checks = pd.read_csv(OUT_DIR / '05x_final_checks.csv')
final_checks.loc[final_checks['check'].eq('review_zip_created'), ['status', 'detail']] = ['PASS' if ZIP_PATH.exists() else 'FAIL', str(ZIP_PATH)]
crit_fail_count = int((final_checks[final_checks['check'] != 'critical_fail_count_zero']['status'] == 'FAIL').sum())
final_checks.loc[final_checks['check'].eq('critical_fail_count_zero'), ['status', 'detail']] = ['PASS' if crit_fail_count == 0 else 'FAIL', str(crit_fail_count)]
final_checks.to_csv(OUT_DIR / '05x_final_checks.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in zip_members:
        if path.exists():
            zf.write(path, path.relative_to(PARK).as_posix())

print('05x patch complete')
print('decision_rows', len(decision))
print('conservative_rows', len(cons))
print('expanded_rows', len(expanded))
print('approval_rows', len(approval))
print('unresolved_rows', len(unresolved))
print('zip_exists', ZIP_PATH.exists())

05x patch complete
decision_rows 91
conservative_rows 22
expanded_rows 82
approval_rows 67
unresolved_rows 7
zip_exists True
